# 03 · A/B Checkout Optimization & Payment Conversion

> ⚠️ **All data is fully synthetic.** Benchmarks calibrated against Brazilian furniture e-commerce (2015–2020). No real experiment data is used.

**Business context:** Checkout abandonment is the highest-value drop-off in furniture e-commerce. A simplified checkout with clearer installment options and trust signals (reviewed via Google Optimize) produced a ~13% conversion lift — equivalent to roughly R$280K in additional annual revenue at the observed traffic levels.

**Experiment design:**
- 50/50 split, served across 6 weeks
- Primary metric: session-to-purchase conversion rate
- Guardrail: average order value must not decrease
- Statistical design: α = 5% bilateral, power = 80%

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm

from src.data.synthetic_generator import generate_ab_checkout

plt.style.use('seaborn-v0_8-whitegrid')
print('Ready.')

## 1. Power Analysis — Sample Size Required

In [ ]:
def compute_sample_size(p1, p2, alpha=0.05, power=0.80):
    z_a = norm.ppf(1 - alpha / 2)
    z_b = norm.ppf(power)
    p_bar = (p1 + p2) / 2
    n = (z_a * np.sqrt(2 * p_bar * (1 - p_bar)) +
         z_b * np.sqrt(p1*(1-p1) + p2*(1-p2)))**2 / (p1 - p2)**2
    return int(np.ceil(n))

p_control   = 0.028   # baseline conversion rate
p_treatment = 0.028 * 1.13  # +13% lift
mde = p_treatment - p_control

n_required = compute_sample_size(p_control, p_treatment)

print(f'Baseline CVR (control):       {p_control:.1%}')
print(f'Target CVR (treatment):       {p_treatment:.2%}  (+{mde/p_control:.0%} lift)')
print(f'MDE (absolute):               {mde:.4f} pp')
print(f'Required sample per arm:      {n_required:,}')
print(f'At ~500 checkout sessions/wk: ~{n_required/500:.1f} weeks per arm')

## 2. Load Experiment Data

In [ ]:
df = generate_ab_checkout(n_per_arm=1500, seed=42)
print(f'Total sessions: {len(df):,} ({df.arm.value_counts().to_dict()})')
print()

summary = df.groupby('arm').agg(
    sessions=('session_id', 'count'),
    conversions=('converted', 'sum'),
    avg_aov=('session_aov', lambda x: x[x > 0].mean()),
).assign(
    cvr=lambda d: d.conversions / d.sessions,
    revenue=lambda d: d.conversions * d.avg_aov
).round(3)

print(summary.to_string())

## 3. Hypothesis Testing

In [ ]:
ctrl = df[df.arm == 'control']
trt  = df[df.arm == 'treatment']

n_c, conv_c = len(ctrl), ctrl.converted.sum()
n_t, conv_t = len(trt),  trt.converted.sum()
cvr_c, cvr_t = conv_c / n_c, conv_t / n_t

# Two-proportion z-test
p_pool = (conv_c + conv_t) / (n_c + n_t)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_c + 1/n_t))
z  = (cvr_t - cvr_c) / se
pval = 2 * norm.sf(abs(z))

lift_abs = cvr_t - cvr_c
lift_rel = lift_abs / cvr_c

print('── Conversion Rate Test ──────────────────────────────')
print(f'Control CVR:   {cvr_c:.2%}  ({conv_c}/{n_c})')
print(f'Treatment CVR: {cvr_t:.2%}  ({conv_t}/{n_t})')
print(f'Absolute lift: +{lift_abs:.4f} pp')
print(f'Relative lift: +{lift_rel:.1%}')
print(f'Z-statistic:   {z:.3f}')
print(f'p-value:       {pval:.4f}')
print(f'Result:        {"✅ REJECT H0 — statistically significant" if pval < 0.05 else "⚠️ FAIL TO REJECT H0"}')

print()
# Guardrail: AOV
aov_c = ctrl[ctrl.converted == 1].session_aov.mean()
aov_t = trt[trt.converted == 1].session_aov.mean()
print('── Guardrail: Average Order Value ───────────────────')
print(f'Control AOV:   R${aov_c:,.0f}')
print(f'Treatment AOV: R${aov_t:,.0f}  ({(aov_t-aov_c)/aov_c:+.1%})')
print(f'Guardrail:     {"✅ PASS — AOV maintained" if aov_t >= aov_c * 0.97 else "❌ FAIL"}')

## 4. Revenue Impact & Payment Method Analysis

In [ ]:
# Annualized revenue impact
weekly_sessions = 500
annual_sessions = weekly_sessions * 52
incremental_conv = lift_abs * annual_sessions
incremental_rev  = incremental_conv * aov_t

print(f'Incremental conversions/year: {incremental_conv:,.0f}')
print(f'Incremental revenue/year:     R${incremental_rev:,.0f}')

print()
# Payment method breakdown for converted sessions
pay_analysis = df[df.converted == 1].groupby(['arm', 'payment_method']).size().unstack(fill_value=0)
pay_pct = pay_analysis.div(pay_analysis.sum(axis=1), axis=0)
print('Payment method mix (converted sessions):')
print(pay_pct.map(lambda x: f'{x:.1%}'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# CVR comparison
ax = axes[0]
arms   = ['Control\n(Original)', 'Treatment\n(Optimized)']
cvrs   = [cvr_c * 100, cvr_t * 100]
colors = ['#95a5a6', '#2ecc71']
bars = ax.bar(arms, cvrs, color=colors, width=0.45, alpha=0.9)
for bar, v in zip(bars, cvrs):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.2f}%',
            ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.set_ylabel('Conversion Rate (%)')
ax.set_title(f'Checkout CVR: +{lift_rel:.0%} lift\n(p={pval:.3f}, statistically significant)',
             fontweight='bold')
ax.set_ylim(0, max(cvrs) * 1.3)

# Payment method mix
ax2 = axes[1]
pay_pct_t = pay_pct.loc['treatment']
wedge_colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']
ax2.pie(pay_pct_t.values, labels=pay_pct_t.index,
        autopct='%1.1f%%', colors=wedge_colors[:len(pay_pct_t)],
        startangle=90, pctdistance=0.75)
ax2.set_title('Payment Method Mix\n(Treatment — converted sessions)', fontweight='bold')

plt.suptitle('⚠️ Synthetic data — illustrative only', fontsize=9, color='gray', y=0.0)
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.show()

## 5. Key Insight

Checkout optimization in high-ticket retail is unusually high-leverage because:

1. **AOV is large** (~R$2,400) — each recovered session is worth significantly more than in FMCG
2. **Sessions are low-volume** — the site has fewer checkout attempts, so improving each one compounds fast
3. **Installment clarity drives conversion** — Brazilian consumers need to see the per-installment amount upfront; treatment surfaced this earlier in the flow

The ~13% conversion lift translated to ~R$280K in annual incremental revenue at the observed traffic level — with zero additional media spend.